# GDELT GKG Data collection

## 1. Setup

In [1]:
import pandas as pd
import requests
import os

In [2]:
def show_error(code, url):
    print("Error", code, "for",url)

In [3]:
def download_gkg(url):
    pieces = url.split("/")
    path = "data/" + pieces[-1]
    r = requests.get(url)
    if r.status_code == 200:
        with open(path, "wb") as f:
            f.write(r.content)
        if os.path.getsize(path) != len(r.content):
            print("Size mismatch for", path, "Expected", len(r.content), "got", os.path.getsize(path))
    else:
        show_error(r.status_code, url)

In [4]:
def gkg_url(date):
    return "http://data.gdeltproject.org/gdeltv2/" + date.strftime("%Y%m%d%H%M%S") +  ".gkg.csv.zip"

In [5]:
def extract_title(text):
    if pd.isna(text) or "<PAGE_TITLE>" not in text:
        return None
    else:
        after = text.split("<PAGE_TITLE>")[1]
        title = after.split("</PAGE_TITLE>")[0]
        return title

In [6]:
def company_pattern(names):
    pattern = r"\b(?:" + "|".join(names) + r")\b"
    return pattern

In [7]:
companies = {
    "LVMH": ["lvmh", "moet hennessy", "moët hennessy"],
    "TotalEnergies": ["totalenergies", "total energies"],
    "Airbus": ["airbus"],
    "Kering": ["kering"],
    "Schneider Electric": ["schneider electric"],
    "Capgemini": ["capgemini"],
    "Danone": ["danone"],
    "Pernod Ricard": ["pernod ricard"],
    "Stellantis": ["stellantis"],
    "BNP Paribas": ["bnp paribas", "bnp"],
    "Société Générale": ["societe generale", "société générale", "socgen"],
    "Safran": ["safran"],
    "Sanofi": ["sanofi"],
    "Air Liquide": ["air liquide"],
}

## 2. Download

In [8]:
slots = pd.date_range("2024-10-16 08:00", "2024-10-16 10:00", freq="15min")

In [9]:
for slot in slots:
    download_gkg(gkg_url(slot))

## 3. Exploration

In [10]:
gkg = pd.read_csv("data/20241016083000.gkg.csv.zip", sep="\t", header=None, dtype=str, quoting=3)
print(gkg.shape)
print(gkg[1].nunique(), gkg[1].iloc[0])

(1662, 27)
1 20241016083000


In [11]:
row = gkg.iloc[0]
for i, val in enumerate(row):
    print(i, len(str(val)), str(val)[:80])

0 16 20241016083000-0
1 14 20241016083000
2 1 1
3 18 lasvegasweekly.com
4 52 https://lasvegasweekly.com/events/2024/oct/16/57381/
5 3 nan
6 3 nan
7 3 nan
8 3 nan
9 3 nan
10 3 nan
11 9 las vegas
12 13 Las Vegas,429
13 3 nan
14 3 nan
15 88 6.93069306930693,9.9009900990099,2.97029702970297,12.8712871287129,21.7821782178
16 3 nan
17 5451 wc:80,c1.2:2,c12.1:6,c12.10:11,c12.12:6,c12.13:3,c12.14:4,c12.5:6,c12.7:6,c12.8:
18 79 https://media.lasvegasweekly.com/img/events/2024/10/Need-for-Speed-1200x628.jpg
19 3 nan
20 3 nan
21 3 nan
22 3 nan
23 66 Every Monday,13;South Point,354;South Point Yeti,415;Las Vegas,469
24 73 500,points on slots,55;100,dollars free play,204;400,Unisex Pullover,275;
25 3 nan
26 258 <PAGE_LINKS>https://lasvegasweekly.com/events/2024/oct/16/57381/%20https://south


In [12]:
print(gkg[13].head(10))

0                                                  NaN
1       nottingham university;university of nottingham
2                                                  NaN
3                                                  NaN
4                                  campaign collection
5          insurance technology market;qurate research
6    co louth;commission for the location of victim...
7    kerala high court on wednesday;high court;oppo...
8    parliament labour mp kim leadbeater;royal comm...
9                                          white house
Name: 13, dtype: str


In [13]:
print(gkg[13].notna().sum())

1202


In [14]:
print(gkg[26].iloc[0])

<PAGE_LINKS>https://lasvegasweekly.com/events/2024/oct/16/57381/%20https://southpointcasino.com/casino/slots/the-need-for-speed-south-point-400-giveaway/</PAGE_LINKS><PAGE_TITLE>Events Calendar - The Need For Speed: South Point 400 Gift Giveaway</PAGE_TITLE>


In [15]:
for file in sorted(os.listdir("data")):
    df = pd.read_csv("data/" + file, header=None, dtype=str, quoting=3, sep="\t")
    mask_org = df[13].str.contains("lvmh", na=False)  
    mask_col26 = df[26].str.contains("lvmh", case=False, na=False) 
    print(file, mask_org.sum(), mask_col26.sum())

20241016080000.gkg.csv.zip 0 1
20241016081500.gkg.csv.zip 0 3
20241016083000.gkg.csv.zip 0 1
20241016084500.gkg.csv.zip 0 0
20241016090000.gkg.csv.zip 0 0
20241016091500.gkg.csv.zip 0 1
20241016093000.gkg.csv.zip 0 1
20241016094500.gkg.csv.zip 0 0
20241016100000.gkg.csv.zip 0 1


In [16]:
df = pd.read_csv("data/20241016081500.gkg.csv.zip", sep="\t", header=None, dtype=str, quoting=3)
mask_col26 = df[26].str.contains("lvmh", na=False, case=False)

for text in df[26][mask_col26]:
    print(text)
    print()

for orgs in df[13][mask_col26]:
    print(orgs)

<PAGE_LINKS>https://www.thisismoney.co.uk/money/markets/article-13963701/Luxury-giant-LVMH-suffers-quarterly-sales-drop-pandemic.html</PAGE_LINKS><PAGE_AUTHORS>Live Commentary</PAGE_AUTHORS><PAGE_PRECISEPUBTIMESTAMP>20241016064700</PAGE_PRECISEPUBTIMESTAMP><PAGE_TITLE>BUSINESS LIVE: Inflation slows to 1.7%; Whitbread eyes &#xA3;300m profit boost; Just Eat hit by US slump</PAGE_TITLE>

<PAGE_LINKS>https://www.thisismoney.co.uk/money/markets/article-13963701/Luxury-giant-LVMH-suffers-quarterly-sales-drop-pandemic.html</PAGE_LINKS><PAGE_AUTHORS>Live Commentary</PAGE_AUTHORS><PAGE_PRECISEPUBTIMESTAMP>20241016064700</PAGE_PRECISEPUBTIMESTAMP><PAGE_ALTURL_AMP>https://www.thisismoney.co.uk/money/markets/article-13965119/amp/BUSINESS-LIVE-Inflation-slows-Whitbread-eyes-profit-boost-Just-Eat-hit-slump.html</PAGE_ALTURL_AMP><PAGE_TITLE>BUSINESS LIVE: Inflation slows to 1.7%; Whitbread eyes &#xA3;300m profit boost; Just Eat hit by US slump</PAGE_TITLE>

<PAGE_AUTHORS>MarketScreener</PAGE_AUTHORS>

In [17]:
for file in sorted(os.listdir("data")):
    df = pd.read_csv("data/" + file, sep="\t", header=None, dtype=str, quoting=3)
    df["title"] = df[26].apply(extract_title)
    mask_title = df["title"].str.contains("lvmh", na=False, case=False)
    print(file, mask_title.sum())
    for t in df["title"][mask_title]:
        print(file, t)
    

20241016080000.gkg.csv.zip 1
20241016080000.gkg.csv.zip LVMH shares hit by weak Q3 sales weigh on wider sector
20241016081500.gkg.csv.zip 1
20241016081500.gkg.csv.zip LVMH Results Cast Cloud on Luxury Stocks
20241016083000.gkg.csv.zip 1
20241016083000.gkg.csv.zip LVMH sales fall 3% as China demand for luxury goods worsens | The Asahi Shimbun: Breaking News, Japan News and Analysis
20241016084500.gkg.csv.zip 0
20241016090000.gkg.csv.zip 0
20241016091500.gkg.csv.zip 0
20241016093000.gkg.csv.zip 1
20241016093000.gkg.csv.zip LVMH plunges as Chinese luxury spending slowdown worsens
20241016094500.gkg.csv.zip 0
20241016100000.gkg.csv.zip 1
20241016100000.gkg.csv.zip LVMH sales fall 3% as China demand for luxury goods worsens


In [18]:
for file in sorted(os.listdir("data")):
    df = pd.read_csv("data/" + file, sep="\t", header=None, dtype=str, quoting=3)
    df["title"] = df[26].apply(extract_title)
    for company, names in companies.items():
        mask_title = df["title"].str.contains(company_pattern(names), na=False, case=False)
        n = mask_title.sum()
        if n > 0:
            for t in df["title"][mask_title]:
                print(file, company, t)

20241016080000.gkg.csv.zip LVMH LVMH shares hit by weak Q3 sales weigh on wider sector
20241016080000.gkg.csv.zip Stellantis Stellantis shipments drop 20 percent in third quarter
20241016080000.gkg.csv.zip Safran Safran Data Systems Expands in Chennai to Drive Aerospace and Automotive Growth
20241016081500.gkg.csv.zip LVMH LVMH Results Cast Cloud on Luxury Stocks
20241016081500.gkg.csv.zip Stellantis Stellantis sees 20% y/y decline in third-quarter shipments By Reuters
20241016083000.gkg.csv.zip LVMH LVMH sales fall 3% as China demand for luxury goods worsens | The Asahi Shimbun: Breaking News, Japan News and Analysis
20241016083000.gkg.csv.zip TotalEnergies Brunei: Hibiscus Petroleum Completes Acquisition of TotalEnergies EP (Brunei)
20241016083000.gkg.csv.zip Airbus Airbus : and Toshiba to partner on superconductivity research
20241016084500.gkg.csv.zip TotalEnergies Senegal/Ghana: Senegal, Uganda and DR Congo Seal TotalEnergies CAF Afcon Places, Sudan Sink Ghana
20241016084500.gkg.c

## 4. Findings: GDELT GKG exploration (16 Oct 2024, 08:00–10:00 UTC)

Test case: LVMH released its Q3 sales on 15 Oct 2024 after the Paris close, and the stock fell sharply on 16 Oct.

- **Organizations column (13) is unreliable**: 0 matches for "lvmh" in 9 files, even for an article titled "LVMH Results Cast Cloud on Luxury Stocks" (GDELT only extracted Burberry and Swatch).
- **Column 26 mixes titles and links**: 8 matches in total, but in the 08:15 file only 1 out of 3 was a real LVMH headline. The other 2 matched a URL.
- **Exact timestamp**: `<PAGE_PRECISEPUBTIMESTAMP>` gives the publication time in UTC, needed to avoid look-ahead bias.
- **Sparse coverage**: very few articles per company, even on a major news day.
- **Duplicates and syndicated news**: the same story can appear several times, e.g. a wire story republished by several outlets ("LVMH sales fall 3% as China demand for luxury goods worsens", found at 08:30 and 10:00). Titles are not always identical (one ends with the site name), so exact matching misses some duplicates. Open question: count a story once (to avoid overweighting its sentiment) or once per republication (as a measure of media attention).
- **News that reports the price move**: titles such as "LVMH plunges..." or "LVMH shares hit..." describe the market reaction itself. Using them to predict the same day's return would leak the outcome into the signal (look-ahead bias).
- **Company matching on titles (14 companies, 9 files)**: 19 matching titles, 15 truly about the company (precision ≈ 79%, small sample). Rules: (1) the company or a subsidiary is the subject; (2) the company is a secondary actor (seller, adviser); (3) unrelated mention or noise.
- **Sponsored events**: company names appear in event names, e.g. a football headline about the "TotalEnergies CAF Afcon" (the same risk is expected with the BNP Paribas Open tennis tournament).
- **Auto-generated articles**: e.g. "Magnus Financial Group LLC Reduces Position in TotalEnergies SE (NYSE:TTE)". No real information, and the wording may bias the sentiment score.
- **Articles vs news events**: the 15 relevant titles cover only 8 distinct news events (e.g. 5 LVMH titles, all about the Q3 sales).
- **Language and encoding**: some titles are in French and contain HTML codes (`jusqu'&#xE0;` instead of "jusqu'à").

### Decisions
1. Match companies on `<PAGE_TITLE>` only, with a list of name variants per company.
2. Remove duplicate articles.
3. Measure coverage on ~10 random days before choosing the modelling approach (stock level on news days, CAC 40 index level, or French-language sources).
4. Timing rule (baseline): the signal for day t only uses news published before the Paris close (17:30) on day t, and predicts the return from close t to close t+1, as in Kirtac & Germano. Robustness test: use news published before the open (09:00) to predict the open-to-close return of day t. GDELT timestamps are in UTC, so conversions must account for daylight saving time.
5. Decide how to treat syndicated news, and document the choice.
6. Universe: 14 CAC 40 companies, chosen for unambiguous names, international coverage, sector diversity and continuous index membership (to be checked on Euronext).
7. Match whole words only (`\b`), with company names but no brands in the first version; measure recall later.
8. Remove auto-generated holdings articles and sponsored-event mentions.